# Load Fine tuned Llama for Embeddings using Unsloth (Memory Optimized)

In [2]:
# %%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()

# Must install unsloth on Colab first
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-mu5uqpgk/unsloth_462361d61f2c46a9865f98408af25093
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-mu5uqpgk/unsloth_462361d61f2c46a9865f98408af25093
  Resolved https://github.com/unslothai/unsloth.git to commit 677ec0cc20bf7cb4735385c51a22999a64839a83
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from unsloth import FastLanguageModel
import torch
import gc
from transformers import AutoTokenizer
from peft import PeftModel

# 1. Clean up before loading
gc.collect()
torch.cuda.empty_cache()

# 2. Reduce Sequence Length for partial loading (saves memory)
# For embeddings, we usually don't need full 8k context.
# 512 or 1024 is usually sufficient for sentences.
max_seq_length = 1024
dtype = None
load_in_4bit = True
model_config = {"model_name": "unsloth/llama-3-8b", "load_in_4bit": False}

# # Authenticate (required for gated Llama-3 model)
# from huggingface_hub import login
# login()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch
from transformers import AutoTokenizer

# 1. Configuration
cpt_model_path = "/content/drive/MyDrive/SiDiaC-v2/Llama_v02_Final"
base_model_name = "unsloth/meta-llama-3.1-8b" # Load the standard base
max_seq_length = 512
dtype = torch.bfloat16
load_in_4bit = True

print("Step 1: Loading base model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# --- THE FIX: RESIZE TO MATCH ADAPTER ---
print("Step 2: Resizing embeddings to match CPTed vocabulary...")
# We load the tokenizer from your CPT path to get the expanded count
cpt_tokenizer = AutoTokenizer.from_pretrained(cpt_model_path)
model.resize_token_embeddings(len(cpt_tokenizer))
# Update the tokenizer in use to the expanded one
tokenizer = cpt_tokenizer

print("Step 3: Merging PEFT adapters...")
# Manually load the adapters onto the correctly-sized model
model = PeftModel.from_pretrained(model, cpt_model_path, adapter_name="full_cpt")
# 4. Set to Evaluation Mode
model.eval()

print(f"✅ CPTed Model successfully loaded with shape {model.model.model.embed_tokens.weight.shape}")

Step 1: Loading base model...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.


Step 2: Resizing embeddings to match CPTed vocabulary...
Step 3: Merging PEFT adapters...


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


✅ CPTed Model successfully loaded with shape torch.Size([128256, 4096])


In [6]:
# Sample Sinhala Text
text = "මේක සිංහල භාෂාවෙන් ලියවුනු වාක්‍යයක්."
print(f"Input: {text}")

# Tokenize
inputs = tokenizer(text, return_tensors="pt").to("cuda")

# Generate Embeddings
FastLanguageModel.for_inference(model)

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

last_hidden_state = outputs.hidden_states[-1]
print(f"Embeddings Shape: {last_hidden_state.shape}")

`use_return_dict` is deprecated! Use `return_dict` instead!


Input: මේක සිංහල භාෂාවෙන් ලියවුනු වාක්‍යයක්.


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Embeddings Shape: torch.Size([1, 69, 4096])


In [7]:
def get_word_level_embeddings(sentence):
    # 1. Tokenize and get word_ids
    inputs = tokenizer(sentence, return_tensors="pt", add_special_tokens=True).to("cuda")
    word_ids = inputs.word_ids()

    # 2. Get the raw token embeddings from the model
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        # Shape: [batch_size, sequence_length, hidden_size]
        # We take [0] to remove the batch dimension since batch_size=1
        token_embeddings = outputs.hidden_states[-1][0]

    # 3. Group sub-tokens by word_id
    word_to_vectors = {}
    word_to_token_ids = {}

    input_ids = inputs["input_ids"][0]

    for i, word_id in enumerate(word_ids):
        if word_id is None: # Skip special tokens like <s>
            continue

        if word_id not in word_to_vectors:
            word_to_vectors[word_id] = []
            word_to_token_ids[word_id] = []

        word_to_vectors[word_id].append(token_embeddings[i])
        word_to_token_ids[word_id].append(input_ids[i])

    # 4. Average vectors and map to decoded words
    final_embeddings = {}

    # We iterate by sorted word_id to maintain order if needed, or just dict order
    for word_id in sorted(word_to_vectors.keys()):
        # Average the vectors for sub-words
        vectors = torch.stack(word_to_vectors[word_id])
        avg_vec = vectors.mean(dim=0)

        # Decode the word from its tokens to get the EXACT text the model saw
        # This is safer than sentence.split() which might not match tokenization (e.g. punctuation)
        word_text = tokenizer.decode(word_to_token_ids[word_id]).strip()

        if word_text:
            final_embeddings[word_text] = avg_vec

    return final_embeddings

In [8]:
embeddings = get_word_level_embeddings(text)

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [9]:
for word, vector in embeddings.items():
    print(f"Word: {word}, Vector Shape: {vector.shape}")

Word: ම, Vector Shape: torch.Size([4096])
Word: ේක, Vector Shape: torch.Size([4096])
Word: ස, Vector Shape: torch.Size([4096])
Word: ිං, Vector Shape: torch.Size([4096])
Word: හල, Vector Shape: torch.Size([4096])
Word: භ, Vector Shape: torch.Size([4096])
Word: ාෂ, Vector Shape: torch.Size([4096])
Word: ාව, Vector Shape: torch.Size([4096])
Word: ෙන, Vector Shape: torch.Size([4096])
Word: ්, Vector Shape: torch.Size([4096])
Word: ල, Vector Shape: torch.Size([4096])
Word: ියව, Vector Shape: torch.Size([4096])
Word: ුන, Vector Shape: torch.Size([4096])
Word: ු, Vector Shape: torch.Size([4096])
Word: ව, Vector Shape: torch.Size([4096])
Word: ාක, Vector Shape: torch.Size([4096])
Word: ්‍, Vector Shape: torch.Size([4096])
Word: යයක, Vector Shape: torch.Size([4096])
Word: ්., Vector Shape: torch.Size([4096])


In [10]:
import json
from collections import defaultdict
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

In [11]:
with open("/content/drive/MyDrive/LSD-Analysis/Data/century_tokens.json", "r", encoding="utf-8") as f:
    century_tokens_loaded = json.load(f)

In [12]:
century_embeddings = defaultdict(list)

In [13]:
def get_word_vecs(word_list, max_len=510, window_step=300):
    # 0. Sanitize Input & Handle Strings
    if isinstance(word_list, str):
        # Automatically split strings into tokens
        word_list = word_list.split()

    if not isinstance(word_list, list):
        return []

    # Ensure all elements are strings and remove None/empty
    clean_word_list = [str(w).strip() for w in word_list if w is not None and str(w).strip()]

    if not clean_word_list:
        return []

    # 1. Tokenize using is_split_into_words=True
    try:
        full_inputs = tokenizer(clean_word_list, is_split_into_words=True, return_tensors="pt", add_special_tokens=False)
    except Exception as e:
        print(f"Tokenization failed for input head: {clean_word_list[:5]}... Error: {e}")
        return []

    input_ids = full_inputs["input_ids"][0]
    total_tokens = len(input_ids)

    # Storage for all token-level embeddings
    all_token_embeddings = torch.zeros((total_tokens, model.config.hidden_size))
    processed_counts = torch.zeros(total_tokens)

    # Slide the window across the token IDs
    for start_idx in range(0, total_tokens, window_step):
        end_idx = min(start_idx + max_len, total_tokens)

        # Prepare chunk with special tokens <s> and </s>
        chunk_ids = input_ids[start_idx:end_idx]
        chunk_with_special = torch.cat([
            torch.tensor([tokenizer.bos_token_id]),
            chunk_ids,
            torch.tensor([tokenizer.eos_token_id])
        ]).unsqueeze(0)

        # Move to GPU
        chunk_with_special = chunk_with_special.to(model.device)

        with torch.no_grad():
            # CausalLM returns hidden_states if requested
            outputs = model(chunk_with_special, output_hidden_states=True)
            # Get last layer hidden states: [batch, seq, hidden]
            chunk_embeddings = outputs.hidden_states[-1][0, 1:-1, :]

        # Add to global storage (move back to CPU)
        all_token_embeddings[start_idx:end_idx] += chunk_embeddings.cpu()
        processed_counts[start_idx:end_idx] += 1

    # Average overlapping regions
    final_token_embeddings = all_token_embeddings / processed_counts.unsqueeze(1)

    # Map back to Word IDs
    word_ids = full_inputs.word_ids()
    word_vecs = []
    temp_storage = defaultdict(list)

    for i, word_id in enumerate(word_ids):
        if word_id is not None:
            temp_storage[word_id].append(final_token_embeddings[i])

    # Reconstruct in order of original word_list (using safe indexing)
    for word_idx in range(len(clean_word_list)):
        if word_idx in temp_storage:
            word_vec = torch.stack(temp_storage[word_idx]).mean(dim=0)
            word_vecs.append(word_vec)
        else:
            # Fallback for empty/special words using zeros
            word_vecs.append(torch.zeros(model.config.hidden_size))

    return word_vecs

In [14]:
for century, sentences in century_tokens_loaded.items():
    print(f"Processing century: {century}")

    # Added tqdm for progress tracking
    for sentence_item in tqdm(sentences, desc=f"Century {century}", unit="sent"):
        try:
            # Get vectors (returns list of Tensors)
            vectors = get_word_vecs(sentence_item)

            if not vectors: # Skip if result is empty
                continue

            # Post-process input words for storage
            if isinstance(sentence_item, str):
                 clean_sent_list = sentence_item.split()
            elif isinstance(sentence_item, list):
                 clean_sent_list = [str(w).strip() for w in sentence_item if w is not None and str(w).strip()]
            else:
                continue

            # Accumulate vectors DIRECTLY (Keep as Tensors, no .tolist needed for torch.save)
            # We DO move to CPU to save GPU VRAM, but keep as Tensor
            sentence_export = []
            for word, vec in zip(clean_sent_list, vectors):
                sentence_export.append([word, vec.cpu()]) # Ensure on CPU

            # Accumulate in the global dictionary
            century_embeddings[century].append(sentence_export)

        except Exception as e:
            print(f"Error processing sentence: {e}")

Processing century: 5


Century 5:   0%|          | 0/31 [00:00<?, ?sent/s]

Processing century: 12


Century 12:   0%|          | 0/62 [00:00<?, ?sent/s]

Processing century: 13


...

Century 18:   0%|          | 0/238 [00:00<?, ?sent/s]

Processing century: 19


Century 19:   0%|          | 0/872 [00:00<?, ?sent/s]

Processing century: 20


Century 20:   0%|          | 0/1529 [00:00<?, ?sent/s]

In [15]:
word, vector = century_embeddings["5"][0][0]
print(f"\nExample: '{word}' in century 5 has vector of shape {vector.shape}")


Example: 'කාම' in century 5 has vector of shape torch.Size([4096])


In [16]:
century_embeddings["5"]

[[['කාම',
   tensor([-0.3109,  0.6562, -0.5439,  ...,  0.8044, -0.5428, -0.3699])],
  ['ලොක',
   tensor([-0.1161,  0.6670,  0.3704,  ...,  0.3019, -0.1426,  0.6120])],
  ['රූප',
   tensor([-0.8739,  0.4053,  0.5036,  ...,  0.1973, -0.0222,  0.3014])],
  ['ලොක',
   tensor([ 0.0595,  1.0986, -0.4800,  ...,  0.6419,  0.2973,  0.7002])],
  ['අරූපලෝක',
   tensor([ 0.0714,  0.6131, -0.0126,  ...,  0.4415,  0.2486,  0.5855])],
  ['යන්',
   tensor([ 0.7539, -0.6229, -0.0895,  ..., -0.2736, -0.8587, -0.4103])],
  ['ලොකත්රයට',
   tensor([ 0.3109,  0.5779,  0.1505,  ..., -0.3563, -0.1617, -0.6872])],
  ['ශ්රෙෂ්ඨ',
   tensor([ 0.2760,  0.7088, -0.3800,  ...,  0.0670,  0.1456, -0.2819])],
  ['වූ', tensor([-1.3281,  1.2408, -0.1531,  ...,  0.2729,  0.1289, -0.0176])],
  ['මුනීන්ද්ර',
   tensor([ 0.2121,  0.3841,  0.3043,  ..., -0.2449, -0.3806,  0.0716])],
  ['නම්',
   tensor([ 0.1038,  1.4456,  0.5195,  ...,  0.4510, -0.7403, -0.1654])],
  ['වූ', tensor([-0.2715,  0.7717, -0.1699,  ...,  0.6372, -0

In [17]:
output_file = "/content/drive/MyDrive/LSD-Analysis/Data/Llama_NO_EXPA_All_Century_Embeddings.pt"

In [18]:
torch.save(century_embeddings, output_file)